# 데이터 전처리 (Preprocessing)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 데이터 클렌징

In [5]:
# 결측치 처리 - 평균값/최빈값/상수값 대체
from sklearn.impute import SimpleImputer

df = pd.DataFrame({
    "Age" : [25, None, 27, None, 27],
    "Income" : [50_000, 60_000, None, 52_000, 58_000]
})

# imputer = SimpleImputer(strategy='mean')
# imputer = SimpleImputer(strategy='most_frequent')
imputer = SimpleImputer(strategy='constant', fill_value=0)
df = pd.DataFrame(imputer.fit_transform(df), columns=["Age", "Income"])
df

,Age,Income
0,25.0,50000.0
1,0.0,60000.0
2,27.0,0.0
3,0.0,52000.0
4,27.0,58000.0


In [7]:
# 이상치 처리 - 트리기반 이상치 탐지 기법
from sklearn.ensemble import IsolationForest

X = np.array([[1], [2], [2.5], [3], [100]])

iso = IsolationForest()
pred = iso.fit_predict(X)   # 정상 데이터 1, 이상치 데이터 -1
print(pred)

X_cleaned = X[pred == 1]
print(X_cleaned)

[ 1  1  1  1 -1]
[[1. ]
 [2. ]
 [2.5]
 [3. ]]


## 데이터 인코딩

In [10]:
# 라벨 인코딩
from sklearn.preprocessing import LabelEncoder

items = ['TV', '냉장고', '전자렌지', '컴퓨터', '선풍기', '선풍기', '믹서', '믹서']

encoder = LabelEncoder()
labels = encoder.fit_transform(items)   # fit: 범주형 데이터 정렬/인덱싱, transform: 변환

print(labels)
print(encoder.classes_)

# 인코딩 된 숫자 값 되돌리기
encoder.inverse_transform([2,3,1])

[0 1 4 5 3 3 2 2]
['TV' '냉장고' '믹서' '선풍기' '전자렌지' '컴퓨터']


array(['믹서', '선풍기', '냉장고'], dtype='<U4')

In [15]:
# One-hot Encoding
# 범주형 변수마다 하나의 열을 생성하고 해당 범주에만 1을 표시하는 인코딩
from sklearn.preprocessing import OneHotEncoder

items = ['TV', '냉장고', '전자렌지', '컴퓨터', '선풍기', '선풍기', '믹서', '믹서']

# 2차원 데이터로 변경
items = np.array(items).reshape(-1, 1)

encoder = OneHotEncoder()
labels = encoder.fit_transform(items)

print(labels)    # CSR 희소배열 
print(labels.toarray())

print(encoder.categories_)      # 컬럼 순서

pd.DataFrame(labels.toarray(), columns=encoder.categories_)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 8 stored elements and shape (8, 6)>
  Coords	Values
  (0, 0)	1.0
  (1, 1)	1.0
  (2, 4)	1.0
  (3, 5)	1.0
  (4, 3)	1.0
  (5, 3)	1.0
  (6, 2)	1.0
  (7, 2)	1.0
[[1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]]
[array(['TV', '냉장고', '믹서', '선풍기', '전자렌지', '컴퓨터'], dtype='<U4')]


,TV,냉장고,믹서,선풍기,전자렌지,컴퓨터
0,1.0,0.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,0.0,1.0,0.0,0.0
5,0.0,0.0,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0,0.0,0.0
7,0.0,0.0,1.0,0.0,0.0,0.0


## 피처 스케일링과 정규화
머신러닝에서 **피처 스케일링**은 서로 다른 범위를 가지는 데이터를 공통된 기준으로 맞추기 위한 과정이다.

### 표준화 (Standardization)
- **정의**: 평균을 0, 표준편차를 1로 변환하는 방식
- **공식**:
  $$
  z = \frac{x - \mu}{\sigma}
  $$
  여기서 $ \mu $는 평균, $ \sigma $는 표준편차
- **특징**:
  - 데이터가 **정규분포를 따를 때** 효과적
  - **음수 값도 허용**
  - 이상치(outlier)의 영향을 **덜 받음**


### 정규화 (Normalization, Min-Max Scaling)
- **정의**: 데이터를 0과 1 사이로 스케일링
- **공식**:
  $$
  x_{norm} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
  $$
- **특징**:
  - **특정 범위(보통 0~1)** 로 제한
  - **이상치에 민감**
  - 거리 기반 알고리즘(k-NN, SVM 등)에 자주 사용

### 로버스트 스케일링 (Robust Scaling)
- **정의**: 중앙값(Median)과 사분위범위(IQR)를 이용한 스케일링
- **공식**:
  $$
  x_{scaled} = \frac{x - \text{Median}}{\text{IQR}}
  $$
  여기서 IQR = Q3 - Q1 (제3사분위수 - 제1사분위수)
- **특징**:
  - **이상치에 강건(Robust)**
  - 중앙값과 IQR 기준 → **데이터 분포 왜곡이 적음**
  - **StandardScaler보다 이상치 영향이 적은 대안**
  - 다양한 분포의 데이터에 적용 가능

In [16]:
# 데이터 로딩
from sklearn.datasets import load_iris

iris = load_iris()
print(iris.data)

[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]
 [5.4 3.9 1.7 0.4]
 [4.6 3.4 1.4 0.3]
 [5.  3.4 1.5 0.2]
 [4.4 2.9 1.4 0.2]
 [4.9 3.1 1.5 0.1]
 [5.4 3.7 1.5 0.2]
 [4.8 3.4 1.6 0.2]
 [4.8 3.  1.4 0.1]
 [4.3 3.  1.1 0.1]
 [5.8 4.  1.2 0.2]
 [5.7 4.4 1.5 0.4]
 [5.4 3.9 1.3 0.4]
 [5.1 3.5 1.4 0.3]
 [5.7 3.8 1.7 0.3]
 [5.1 3.8 1.5 0.3]
 [5.4 3.4 1.7 0.2]
 [5.1 3.7 1.5 0.4]
 [4.6 3.6 1.  0.2]
 [5.1 3.3 1.7 0.5]
 [4.8 3.4 1.9 0.2]
 [5.  3.  1.6 0.2]
 [5.  3.4 1.6 0.4]
 [5.2 3.5 1.5 0.2]
 [5.2 3.4 1.4 0.2]
 [4.7 3.2 1.6 0.2]
 [4.8 3.1 1.6 0.2]
 [5.4 3.4 1.5 0.4]
 [5.2 4.1 1.5 0.1]
 [5.5 4.2 1.4 0.2]
 [4.9 3.1 1.5 0.2]
 [5.  3.2 1.2 0.2]
 [5.5 3.5 1.3 0.2]
 [4.9 3.6 1.4 0.1]
 [4.4 3.  1.3 0.2]
 [5.1 3.4 1.5 0.2]
 [5.  3.5 1.3 0.3]
 [4.5 2.3 1.3 0.3]
 [4.4 3.2 1.3 0.2]
 [5.  3.5 1.6 0.6]
 [5.1 3.8 1.9 0.4]
 [4.8 3.  1.4 0.3]
 [5.1 3.8 1.6 0.2]
 [4.6 3.2 1.4 0.2]
 [5.3 3.7 1.5 0.2]
 [5.  3.3 1.4 0.2]
 [7.  3.2 4.7 1.4]
 [6.4 3.2 4.5 1.5]
 [6.9 3.1 4.

In [17]:
print(iris.target)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2]


In [19]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

In [20]:
# 표준 스케일링
standard_scaler = StandardScaler()

iris_scaled = standard_scaler.fit_transform(iris.data)
iris_scaled_df = pd.DataFrame(iris_scaled, columns=iris.feature_names)
iris_scaled_df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,-0.900681,1.019004,-1.340227,-1.315444
1,-1.143017,-0.131979,-1.340227,-1.315444
2,-1.385353,0.328414,-1.397064,-1.315444
3,-1.506521,0.098217,-1.283389,-1.315444
4,-1.021849,1.249201,-1.340227,-1.315444
...,...,...,...,...
145,1.038005,-0.131979,0.819596,1.448832
146,0.553333,-1.282963,0.705921,0.922303
147,0.795669,-0.131979,0.819596,1.053935
148,0.432165,0.788808,0.933271,1.448832


In [21]:
# 최대최소 스케일링
minmax_scaler = MinMaxScaler()

iris_scaled = minmax_scaler.fit_transform(iris.data)
iris_scaled_df = pd.DataFrame(iris_scaled, columns=iris.feature_names)
iris_scaled_df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,0.222222,0.625000,0.067797,0.041667
1,0.166667,0.416667,0.067797,0.041667
2,0.111111,0.500000,0.050847,0.041667
3,0.083333,0.458333,0.084746,0.041667
4,0.194444,0.666667,0.067797,0.041667
...,...,...,...,...
145,0.666667,0.416667,0.711864,0.916667
146,0.555556,0.208333,0.677966,0.750000
147,0.611111,0.416667,0.711864,0.791667
148,0.527778,0.583333,0.745763,0.916667


In [22]:
# Robust 스케일링
robust_scaler = RobustScaler()

iris_scaled = robust_scaler.fit_transform(iris.data)
iris_scaled_df = pd.DataFrame(iris_scaled, columns=iris.feature_names)
iris_scaled_df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,-0.538462,1.0,-0.842857,-0.733333
1,-0.692308,0.0,-0.842857,-0.733333
2,-0.846154,0.4,-0.871429,-0.733333
3,-0.923077,0.2,-0.814286,-0.733333
4,-0.615385,1.2,-0.842857,-0.733333
...,...,...,...,...
145,0.692308,0.0,0.242857,0.666667
146,0.384615,-1.0,0.185714,0.400000
147,0.538462,0.0,0.242857,0.466667
148,0.307692,0.8,0.300000,0.666667


In [26]:
# 이상치가 있는 경우 스케일링 기법 비교
df = pd.DataFrame([[10], [12], [14], [15], [18], [100]], columns=["Original"])

df['StandardScaler'] = StandardScaler().fit_transform(df[['Original']])
df['MinMaxScaler'] = MinMaxScaler().fit_transform(df[['Original']])
df['RobustScaler'] = RobustScaler().fit_transform(df[['Original']])

df

,Original,StandardScaler,MinMaxScaler,RobustScaler
0,10,-0.563829,0.000000,-0.947368
1,12,-0.501756,0.022222,-0.526316
2,14,-0.439683,0.044444,-0.105263
3,15,-0.408647,0.055556,0.105263
4,18,-0.315537,0.088889,0.736842
5,100,2.229453,1.000000,18.000000
